In [6]:
from pathlib import Path
import argparse
import datetime
import sys
import logging
import xarray as xr
import pandas as pd
import pygrib
import importlib

In [18]:
# Set this to the parent directory containing the 32-char subfolders
ROOT = Path("/mnt/metdata/W25-2348/glofas/")

# Only process immediate subfolders. If your gribs live deeper, change the walk logic.
SUBDIR_PATTERN = None   # None means process all immediate directories

# Report gaps larger than this threshold (hours)
GAP_THRESHOLD_HOURS = 24

# Optional: set verbose=True to see debug messages per file
VERBOSE = False

APPLY = True

In [9]:
def find_grib_file(folder: Path) -> Path | None:
    """
    Return a Path to a GRIB file inside folder, or None if none found.
    Preference order: data.grib, data.grb, data.grb2, then any .grib/.grb2.
    """
    for name in ("data.grib", "data.grb", "data.grb2"):
        p = folder / name
        if p.exists():
            return p

    # Any grib-like file
    for pattern in ("*.grib", "*.grb", "*.grb2"):
        matches = list(folder.glob(pattern))
        if matches:
            return matches[0]

    return None


def format_dt(dt: datetime.datetime) -> str:
    """
    Compact datetime format for filenames: YYYYmmddTHHMM
    """
    return dt.strftime("%Y%m%dT%H%M")


def times_with_pygrib(path: Path) -> list[datetime.datetime]:
    """
    Read valid times from a GRIB file using pygrib.
    Returns a list of datetime objects (may be empty).
    """

    times: list[datetime.datetime] = []
    with pygrib.open(str(path)) as grbs:
        for msg in grbs:
        # Common attribute names
            dt = getattr(msg, "validDate", None) or getattr(msg, "validityDate", None)
            if dt:
                times.append(dt)
                continue

            # Fallback to dataDate + dataTime (integers)
            data_date = getattr(msg, "dataDate", None)
            data_time = getattr(msg, "dataTime", None)
            if data_date and data_time is not None:
                ds = str(data_date)
                hour = int(data_time // 100)
                minute = int(data_time % 100)
            try:
                times.append(
                datetime.datetime(
                int(ds[:4]), int(ds[4:6]), int(ds[6:8]), hour, minute
                )
                )
            except Exception:
            # if parsing fails, skip this message
                continue
    return times


def times_with_cfgrib(path: Path) -> list[datetime.datetime]:
    """
    Read times from a GRIB file using xarray + cfgrib.
    Returns a list of datetime objects (may be empty).
    """

    times: list[datetime.datetime] = []
    ds = xr.open_dataset(str(path), engine="cfgrib")

    # Common coordinate names: "time" or "valid_time"
    vals = []
    if "time" in ds.coords:
        vals = ds["time"].values
    elif "valid_time" in ds.coords:
        vals = ds["valid_time"].values

    for v in vals:
        try:
            pdts = pd.to_datetime(v)
            if pdts is not pd.NaT:
                times.append(pdts.to_pydatetime())
        except Exception:
        # ignore values we can't parse
            continue

    ds.close()
    return times


def gather_times(path: Path) -> list[datetime.datetime]:
    """
    Try several backends to obtain a list of datetimes from a grib file.
    Order: pygrib, then xarray+cfgrib. Returns empty list if none available.
    """
    # Try pygrib if installed
    try:
        if importlib.util.find_spec("pygrib") is not None:
            times = times_with_pygrib(path)
            if times:
                return times
    except Exception:
        # swallow and try next backend
        pass

    # Try xarray + cfgrib if installed
    try:
        if (
            importlib.util.find_spec("xarray") is not None
            and importlib.util.find_spec("cfgrib") is not None
            ):
            times = times_with_cfgrib(path)
            if times:
                return times
    except Exception:
        pass

    return []


def propose_new_name(grib: Path) -> tuple[str | None, str | None]:
    """
    Propose a new filename for the grib file based on min/max valid times.
    Returns (newname, error). If an error occurs or times couldn't be determined,
    newname is None and error is a short string.
    """
    times = gather_times(grib)
    if not times:
        return None, "no-times"

    tmin = min(times)
    tmax = max(times)

    if tmin == tmax:
        newname = f"data_{format_dt(tmin)}{grib.suffix}"
    else:
        newname = f"data_{format_dt(tmin)}-{format_dt(tmax)}{grib.suffix}"

    return newname, None


def safe_rename(src: Path, dst: Path) -> Path:
    """
    Rename src -> dest. If dest exists, add numeric suffix to avoid overwriting.
    Honors the global APPLY flag (if False, does not perform rename).
    Returns the final target Path (the candidate or the applied path).
    """
    final = dst
    if final.exists():
        base = final.stem
        suffix = final.suffix
        i = 1
        while True:
            candidate = final.with_name(f"{base}_{i}{suffix}")
            if not candidate.exists():
                final = candidate
                break
            i += 1

    logging.info("%s -> %s", src, final)

    if APPLY:
        src.rename(final)
    return final


def _times_with_cfgrib(path: Path):
    """Return sorted list of datetimes using xarray+cfgrib, or [] if it fails."""
    try:
        import xarray as xr
        import pandas as pd
    except Exception:
        return []
    times = []
    try:
        ds = xr.open_dataset(str(path), engine="cfgrib")
    except Exception:
        return []
    try:
        vals = []
        if "time" in ds.coords:
            vals = ds["time"].values
        elif "valid_time" in ds.coords:
            vals = ds["valid_time"].values
        for v in vals:
            try:
                pdts = pd.to_datetime(v)
                if pdts is not pd.NaT:
                    times.append(pdts.to_pydatetime())
            except Exception:
                # skip unparseable
                continue
    finally:
        try:
            ds.close()
        except Exception:
            pass
    times = sorted(set(times))
    return times


def _times_with_pygrib(path: Path):
    """Return sorted list of datetimes using pygrib, or [] if it fails."""
    try:
        import pygrib
    except Exception:
        return []
    times = []
    try:
        with pygrib.open(str(path)) as grbs:
            for msg in grbs:
                dt = getattr(msg, "validDate", None) or getattr(msg, "validityDate", None)
                if dt:
                    times.append(dt)
                else:
                    data_date = getattr(msg, "dataDate", None)
                    data_time = getattr(msg, "dataTime", None)
                    if data_date and data_time is not None:
                        ds = str(data_date)
                        hh = int(data_time // 100)
                        mm = int(data_time % 100)
                        try:
                            times.append(datetime.datetime(
                                int(ds[:4]), int(ds[4:6]), int(ds[6:8]), hh, mm
                            ))
                        except Exception:
                            # skip invalid entries
                            continue
    except Exception:
        return []
    # unique & sorted
    times = sorted(set(times))
    return times


def read_grib_times(path: Path):
    """Return sorted list of datetimes found in GRIB file via pygrib or cfgrib (fallback)."""
    # try pygrib
    times = _times_with_pygrib(path)
    if times:
        if VERBOSE:
            print(f"pygrib read {len(times)} times from {path}")
        return times
    # fallback to cfgrib
    times = _times_with_cfgrib(path)
    if times:
        if VERBOSE:
            print(f"cfgrib read {len(times)} times from {path}")
        return times
    # nothing found
    if VERBOSE:
        print(f"No times found in {path} (pygrib & cfgrib failed or returned no times).")
    return []

### Run the functions over the sub-folders on MetData

In [4]:
from pathlib import Path
from collections import Counter

root = Path("/mnt/metdata/W25-2348/glofas/") # set to the parent directory containing the 32-char folders
folders = sorted([p for p in root.iterdir() if p.is_dir()])

results = []

for folder in folders:
    grib = find_grib_file(folder)
    if grib is None:
        results.append((folder.name, None, "no-file"))
        continue

    # propose a new name based on times 
    newname, err = propose_new_name(grib)
    if err:
        results.append((folder.name, grib.name, err))
        continue

    dst = grib.with_name(newname)
    final_dst = safe_rename(grib, dst)
    applied_note = "applied" if APPLY else "dry-run"
    results.append((folder.name, grib.name, f"-> {final_dst.name} ({applied_note})"))

# display results 
print(f'APPLY = {APPLY}')
for r in results:
    print(*r)
print('Summary: ', Counter([r[2] for r in results]))

APPLY = True
12ae4b1f14e034380b79b3ce02afa0b5 data.grib -> data_20050601T0000-20051104T0000.grib (applied)
18e565b1893bb9dd0219283528f40934 data.grib -> data_20100101T0000-20100109T0000.grib (applied)
1f9ef7243c55cdde7da92ddf86eb87a0 data.grib -> data_20090701T0000-20090809T0000.grib (applied)
2725e5a418ae518d20fc8926d11af383 data.grib -> data_20071102T0000-20080106T0000.grib (applied)
2c366870447050e07e4b9c8efb6a917b data.grib -> data_20090902T0000-20091109T0000.grib (applied)
3f62e73af45228c82b43923465215210 data.grib -> data_20070101T0000-20070508T0000.grib (applied)
48672eb6eeece7aa1c03b64c1644fe71 data.grib -> data_20081102T0000-20090106T0000.grib (applied)
4cba1eea24ccf381a6a71151c9de1a0a data.grib -> data_20080101T0000-20080608T0000.grib (applied)
568544fed59988078d7ee006af59d0f4 data.grib -> data_20100111T0000-20100119T0000.grib (applied)
5af64f94b7a80e75de8192f8ccb7fcd2 data.grib -> data_20100125T0000-20100202T0000.grib (applied)
5eda404cccb78b7719d802d133674046 data.grib -> d

### Scan the directories and collect date/time intervals

In [10]:
# collect one interval per found GRIB file
records = []   # list of dicts: {folder, file, start, end, status}
root = Path(ROOT)

dirs = sorted([p for p in root.iterdir() if p.is_dir()])
for d in dirs:
    if SUBDIR_PATTERN:
        # a simple filter: you can use fnmatch if you want glob rules
        if len(SUBDIR_PATTERN) == 32 and len(d.name) != 32:
            continue
        # or do fnmatch: from fnmatch import fnmatch ; if not fnmatch(d.name, SUBDIR_PATTERN): continue

    grib = find_grib_file(d)
    if grib is None:
        records.append({"folder": d.name, "file": None, "start": None, "end": None, "status": "no-file"})
        continue

    times = read_grib_times(grib)
    if not times:
        records.append({"folder": d.name, "file": grib.name, "start": None, "end": None, "status": "no-times"})
        continue

    start = min(times)
    end = max(times)
    records.append({"folder": d.name, "file": grib.name, "start": start, "end": end, "status": "ok"})

# quick summary counts
from collections import Counter
print("Scanned", len(dirs), "folders; records:", len(records))
print("Status counts:", Counter(r["status"] for r in records))

Scanned 42 folders; records: 42
Status counts: Counter({'ok': 42})


### Print the results

In [11]:
# Pretty-print results; if pandas available, use it for nicer display.
try:
    import pandas as pd
    df = pd.DataFrame(records)
    # format datetimes for readability
    if "start" in df:
        df["start_str"] = df["start"].apply(lambda x: x.isoformat() if x else "")
        df["end_str"]   = df["end"].apply(lambda x: x.isoformat() if x else "")
    display(df.sort_values(by=["start", "folder"], na_position="last"))
except Exception:
    # fallback to plain printing
    for r in sorted(records, key=lambda x: (x["start"] or datetime.datetime.max)):
        print(r["folder"], r["file"], r["start"], "-", r["end"], "status=", r["status"])

,folder,file,start,end,status,start_str,end_str
30,c5a139d17db05396f86db31b303f01e7,data_20030327T0000-20030408T0000.grib,2003-03-27,2003-04-08,ok,2003-03-27T00:00:00,2003-04-08T00:00:00
19,896b6669dec613060cbc4c09d309321a,data_20030403T0000-20030506T0000.grib,2003-04-03,2003-05-06,ok,2003-04-03T00:00:00,2003-05-06T00:00:00
26,b9d3180c0f6b68d74c3d2a87bdc83b15,data_20030501T0000-20040103T0000.grib,2003-05-01,2004-01-03,ok,2003-05-01T00:00:00,2004-01-03T00:00:00
29,c3f1008ad54f6f78a894fa429b3c508b,data_20040101T0000-20040408T0000.grib,2004-01-01,2004-04-08,ok,2004-01-01T00:00:00,2004-04-08T00:00:00
11,65cc055950488e5f44b522db6012d71d,data_20040401T0000-20040708T0000.grib,2004-04-01,2004-07-08,ok,2004-04-01T00:00:00,2004-07-08T00:00:00
22,9d588b8de28aeba53ef734ab1cb43d12,data_20040701T0000-20041009T0000.grib,2004-07-01,2004-10-09,ok,2004-07-01T00:00:00,2004-10-09T00:00:00
23,9e307b8922f35510a678c68395192c76,data_20050101T0000-20050608T0000.grib,2005-01-01,2005-06-08,ok,2005-01-01T00:00:00,2005-06-08T00:00:00
0,12ae4b1f14e034380b79b3ce02afa0b5,data_20050601T0000-20051104T0000.grib,2005-06-01,2005-11-04,ok,2005-06-01T00:00:00,2005-11-04T00:00:00
34,d82d78566d91d445d9fe6ff50aee0a50,data_20051102T0000-20060106T0000.grib,2005-11-02,2006-01-06,ok,2005-11-02T00:00:00,2006-01-06T00:00:00
25,b75b0c95ec6f5299ea24d9baec81c416,data_20060101T0000-20060408T0000.grib,2006-01-01,2006-04-08,ok,2006-01-01T00:00:00,2006-04-08T00:00:00


### Try opening the other files I've got lying around

In [19]:
def merge_intervals(intervals):
    """
    intervals: list of (start:datetime, end:datetime)
    returns merged list sorted, non-overlapping
    """
    if not intervals:
        return []
    # sort by start
    intervals = sorted(intervals, key=lambda x: x[0])
    merged = []
    cur_start, cur_end = intervals[0]
    for s, e in intervals[1:]:
        # if overlapping or contiguous (e.g. s <= cur_end), extend
        # treat small micro-gaps as gap unless you want tolerance
        if s <= cur_end:
            cur_end = max(cur_end, e)
        else:
            merged.append((cur_start, cur_end))
            cur_start, cur_end = s, e
    merged.append((cur_start, cur_end))
    return merged

# collect 'ok' intervals
intervals = [(r["start"], r["end"]) for r in records if r["status"] == "ok"]
intervals = [i for i in intervals if i[0] is not None and i[1] is not None and i[0] <= i[1]]

merged = merge_intervals(intervals)

print("Found", len(intervals), "valid intervals from files; merged into", len(merged), "intervals.")
if not merged:
    print("No valid intervals to report.")
else:
    # show merged intervals
    for s, e in merged:
        dur = e - s
        print(f"Merged interval: {s.isoformat()} -> {e.isoformat()} (duration: {dur})")

    # find gaps between merged intervals and show gaps > threshold
    thresh = datetime.timedelta(hours=GAP_THRESHOLD_HOURS)
    gaps = []
    for (s1, e1), (s2, e2) in zip(merged, merged[1:]):
        gap_len = s2 - e1
        if gap_len > thresh:
            gaps.append((e1, s2, gap_len))

    print()
    if not gaps:
        print(f"No gaps larger than {GAP_THRESHOLD_HOURS} hours found.")
    else:
        print(f"Gaps > {GAP_THRESHOLD_HOURS} hours:")
        for gstart, gend, glen in gaps:
            days = glen.total_seconds() / 86400.0
            print(f"  Missing from {gstart.isoformat()} to {gend.isoformat()}  (gap = {glen}, ≈ {days:.2f} days)")

Found 42 valid intervals from files; merged into 3 intervals.
Merged interval: 2003-03-27T00:00:00 -> 2004-10-09T00:00:00 (duration: 562 days, 0:00:00)
Merged interval: 2005-01-01T00:00:00 -> 2010-02-20T00:00:00 (duration: 1876 days, 0:00:00)
Merged interval: 2012-01-01T00:00:00 -> 2012-04-08T00:00:00 (duration: 98 days, 0:00:00)

Gaps > 24 hours:
  Missing from 2004-10-09T00:00:00 to 2005-01-01T00:00:00  (gap = 84 days, 0:00:00, ≈ 84.00 days)
  Missing from 2010-02-20T00:00:00 to 2012-01-01T00:00:00  (gap = 680 days, 0:00:00, ≈ 680.00 days)
